# GroundLoop on Colab

Runs the whole project against **real weights**: the three-way ablation before
fine-tuning, LoRA SFT, DPO, then the ablation again with the adapter loaded.

**Runtime → Change runtime type → T4 GPU** before you start. The free tier is enough.

Rough timings on a T4 with a 0.8B model, all estimates:

| Step | Time |
|---|---|
| Setup + install | ~4 min |
| Smoke test (1 question) | ~1 min |
| Ablation before, `--limit 8` | ~5 min |
| Ablation before, full | ~30 min |
| Build training data | ~15 min |
| SFT | ~10 min |
| DPO | ~10 min |
| Ablation after, full | ~30 min |

Most of that is generation, one call at a time. Run the `--limit 8` pass first —
if the model's output breaks the parsers, you want to find out in five minutes,
not thirty.

**Colab runtimes are ephemeral.** The last section saves the adapter and the
results to Drive. Do that before you close the tab, or the run is gone.


## 1. Check the GPU


In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.'
)
gpu = torch.cuda.get_device_name(0)
major, _ = torch.cuda.get_device_capability(0)
# Turing (T4, capability 7.x) has no usable bfloat16. Ampere and later do.
PRECISION = 'fp16' if major < 8 else 'bf16'
DTYPE = 'float16' if major < 8 else 'bfloat16'
print(f'{gpu} -> training in {PRECISION}, generating in {DTYPE}')


## 2. Clone and install

Colab already ships torch, so only the training stack is installed here —
`requirements-train.txt` pins torch too and would trigger a needless re-download.

`pip install -e .` is what makes `python -m groundloop...` work. The package
lives under `src/`; `make` sets `PYTHONPATH` for its own targets, but the
`!python -m` cells below are separate subprocesses and need the package properly
installed. The last line fails loudly if it did not take.

The loop, the retriever and the evaluation are standard-library Python; none of
the training stack is needed for `make test`.


In [ ]:
!git clone --branch claude/groundloop-retrieval-critique-gebq3b \
    https://github.com/Mars-2030/DoubtLoop.git
%cd DoubtLoop
!pip install -q 'transformers>=4.51' 'trl>=0.14' 'peft>=0.14' 'datasets>=3.0' 'accelerate>=1.0'

# The package lives under src/, so install it (editable, and it declares no
# dependencies of its own). Every `!python -m groundloop...` cell below runs in
# its own subprocess, and a path set only for this kernel would not reach them.
!pip install -q -e .

# Colab preinstalls an old torchao. peft probes for it while building the LoRA
# layers and raises on a version below 0.16 instead of treating it as absent,
# which kills the SFT cell. Nothing here uses torchao, so remove it; peft then
# finds no module and moves on.
!pip uninstall -y -q torchao 2>/dev/null || true

import os
os.environ['PYTHONPATH'] = os.getcwd() + '/src'   # belt and braces
!python -c "import groundloop; print('groundloop', groundloop.__version__, 'importable')"


## 3. Sanity check before spending GPU time

The test suite needs no GPU, no network and no weights. If it fails, the problem
is the checkout, not the model.


In [ ]:
!make test


## 4. Pick a model

`MODEL` is the target from `PLAN.md`. **Confirm the repo id resolves** — if it
404s, the id is wrong or the model is gated, and the next cell will say which.
Any small instruct model with a chat template works; tool-calling support makes
the GroundLoop condition better but is not required, because the loop falls back
to a harness-built query and reports how often it had to.

For a gated model (Gemma, for instance) accept the licence on its model page and
add your token: `from huggingface_hub import login; login('hf_...')`.


In [ ]:
MODEL = 'Qwen/Qwen3.5-0.8B-Instruct'

FALLBACKS = [
    'Qwen/Qwen3-0.6B',
    'Qwen/Qwen2.5-0.5B-Instruct',
    'HuggingFaceTB/SmolLM2-360M-Instruct',
]

from huggingface_hub import model_info
from huggingface_hub.utils import HfHubHTTPError

def resolve(candidates):
    for repo in candidates:
        try:
            model_info(repo)
            return repo
        except HfHubHTTPError as exc:
            print(f'{repo}: {exc.response.status_code}')
    return None

MODEL = resolve([MODEL] + FALLBACKS)
assert MODEL, 'none of the candidate model ids resolved - set MODEL by hand'
print('using', MODEL)


## 5. First contact: one question, three conditions

This is the cell that matters most. The pipeline has been validated against a
deterministic stand-in, not against a real model, so the likeliest failure is
format handling — the chat template, the shape of the tool call, whether the
critique comes back as JSON.

What you want to see: the base draft says **24 hours** (the terrestrial prior),
and the GroundLoop answer says **31** and cites `[f03]`. If the GroundLoop column
shows `(harness fallback)` next to the search line, the model did not emit a
usable tool call and the harness built the query instead — worth knowing before
you interpret anything downstream.


In [ ]:
!python demo/app.py --cli --backend transformers --model $MODEL --dtype $DTYPE \
    "How many hours are there in a KS-9 station day?"


## 6. The ablation, before any fine-tuning

Short pass first. If the numbers are absurd — every condition at 100%, or no
claims extracted at all — stop and look at a transcript rather than running the
full set.


In [ ]:
!python -m groundloop.eval.run_all --backend transformers --model $MODEL \
    --dtype $DTYPE --limit 8 --out-dir results-smoke --no-save-trajectories


In [ ]:
# Full run. ~30 minutes; the trajectories are saved so you can re-score later
# without regenerating.
!python -m groundloop.eval.run_all --backend transformers --model $MODEL \
    --dtype $DTYPE --out-dir results-before


### How robust is it before training?

Optional, ~15 minutes. Retrieval stress is the more informative of the two here:
an untrained small model handed evidence that cannot answer the question is
exactly where fabrication shows up.


In [ ]:
!python -m groundloop.eval.retrieval_stress --backend transformers --model $MODEL \
    --dtype $DTYPE --out-dir results-before


## 7. Build training data from the real model

This re-runs the loop with the real model and keeps only the trajectories where
the revision actually beat the draft by the harness's own measure.

**Read the yield line.** It is the most informative number in this notebook. A
real model paraphrases, and the lexical support check is at its weakest on
paraphrase, so a near-zero yield is as likely to be the metric rejecting good
revisions as the model producing bad ones. If yield is low the script names the
two levers: `--no-require-correct` (keep grounded revisions that miss the
reference answer keys) and `--tau 0.5` (loosen the *selection* threshold only —
the evaluation still scores at the reporting threshold, so this trades
training-data purity for volume rather than flattering the result).

Fewer records than the effective batch means the trainer takes one optimizer step
per epoch and learns nothing. Section 8 refuses to run in that case rather than
producing an adapter that does nothing.


In [ ]:
!python -m groundloop.data_gen.build_trajectories --backend transformers \
    --model $MODEL --dtype $DTYPE

!head -c 600 data/generated/sft.jsonl; echo; echo '---'
!wc -l data/generated/*.jsonl


In [ ]:
# Only if the yield above was low. Loosens selection, not reporting.
# !python -m groundloop.data_gen.build_trajectories --backend transformers \
#     --model $MODEL --dtype $DTYPE --no-require-correct --tau 0.5


## 8. SFT (LoRA)

Teaches the format: see a question, call `search`, answer out of what came back,
with citations. `--fp16` is set automatically on a T4 — without it the run falls
back to fp32 and takes several times longer.


In [ ]:
FP16 = '--fp16' if PRECISION == 'fp16' else ''

!python -m groundloop.train.sft --model $MODEL --output-dir outputs/sft-lora \
    $FP16 --epochs 3 --batch-size 1 --grad-accum 16 --max-seq-len 2048


## 9. DPO

Teaches the preference: the grounded revision over the original draft, both
answering one prompt that already contains the evidence. The SFT adapter is
merged in first so the reference model stays well defined.


In [ ]:
!python -m groundloop.train.dpo --model $MODEL --adapter outputs/sft-lora \
    --output-dir outputs/dpo-lora $FP16 --batch-size 1 --grad-accum 16 \
    --max-length 2048 --max-prompt-length 1536


## 10. The ablation again, with the adapter

Same command as before plus `--adapter`. This is the before/after the project is
actually about.


In [ ]:
!python -m groundloop.eval.run_all --backend transformers --model $MODEL \
    --dtype $DTYPE --adapter outputs/dpo-lora --out-dir results-after


## 11. Compare


In [ ]:
import json

def headline(path):
    payload = json.load(open(path))
    return {c: (v['qa']['hallucination_rate'], v['qa']['accuracy'],
                v['sycophancy']['pushback_rate'])
            for c, v in payload['conditions'].items()}

before, after = headline('results-before/metrics.json'), headline('results-after/metrics.json')

print(f"{'condition':<16} {'hallucination':>22} {'accuracy':>18} {'pushback':>18}")
print(f"{'':16} {'before':>10} {'after':>11} {'before':>8} {'after':>9} {'before':>8} {'after':>9}")
for c in ('base', 'plain_critique', 'groundloop'):
    b, a = before[c], after[c]
    print(f'{c:<16} {b[0]:>9.1f}% {a[0]:>10.1f}% {b[1]:>7.1f}% {a[1]:>8.1f}% {b[2]:>7.1f}% {a[2]:>8.1f}%')

print()
print(open('results-after/comparison_table.md').read())


## 12. Save before the runtime dies

Colab reclaims the machine and everything above goes with it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/groundloop
!cp -r outputs results-before results-after /content/drive/MyDrive/groundloop/
!ls -R /content/drive/MyDrive/groundloop | head -30


In [ ]:
# Or download a zip instead.
!zip -qr groundloop-run.zip outputs results-before results-after
from google.colab import files
files.download('groundloop-run.zip')


## Troubleshooting

**`ModuleNotFoundError: No module named 'groundloop'`.** Section 2 did not run,
or the runtime restarted and dropped the install. Re-run it — or, from the repo
root, `!pip install -q -e .` on its own is enough. As an instant alternative
without pip: `import os; os.environ['PYTHONPATH'] = '/content/DoubtLoop/src'`,
which the `!` subprocesses inherit.

**`ImportError: Found an incompatible version of torchao`** during SFT. Colab
preinstalls torchao 0.10; peft probes for it while building the LoRA layers and
raises on anything below 0.16 rather than treating it as absent. Section 2
removes it. If you skipped that: `!pip uninstall -y torchao`, then re-run the SFT
cell. Upgrading (`!pip install -U torchao`) also works but risks a torch version
mismatch, and nothing here uses it.

**SFT refuses to start, saying the dataset is too small.** Working as intended —
see the yield discussion in section 7. Four records at an effective batch of 16
is three optimizer steps, which trains nothing while looking like a successful
run in the logs. Regenerate with `--no-require-correct`, or lower `--grad-accum`,
or pass `--allow-tiny-dataset` if a token run is genuinely what you want.

**CUDA out of memory.** Drop `--max-seq-len` to 1024, keep `--batch-size 1`, and
raise `--grad-accum` to hold the effective batch. If it still OOMs the model is
too big for a T4 at this sequence length — the trajectory-style SFT records are
long because they carry the retrieved passages.

**Generation is very slow.** Expected: this is batch-1 `transformers.generate`.
For a faster path install vLLM (`pip install vllm`), run
`vllm serve $MODEL --enable-auto-tool-choice --tool-call-parser hermes &` in a
cell, wait for it to come up, then swap every `--backend transformers --dtype ...`
for `--backend openai --base-url http://localhost:8000/v1`. The install costs
several minutes and vLLM plus a training run in one runtime will fight over VRAM,
so it is only worth it if you are evaluating repeatedly and not training.

**`harness fallback` on every search.** The model is not emitting a parseable tool
call. Check `results-before/comparison_table.md` under *Tool use* for the exact
rate. The loop still works — it builds the query itself — but the tool-use numbers
then describe the harness, not the model. Try `--thinking`, or a model with
stronger tool-calling.

**Every claim scores as unsupported.** The lexical scorer is strict about
paraphrase. Sanity-check by re-scoring at a lower threshold:
`python -m groundloop.eval.sensitivity --backend transformers --model $MODEL
--dtype $DTYPE`. If the ordering only holds at one threshold, that is a finding
about the metric and belongs in the write-up.

**The session died mid-run.** Re-run from section 2, then skip to whichever step
you had saved to Drive.

---

### One caveat worth carrying into the write-up

There are 39 QA items and 12 probes, and the training data derived from them is
smaller still. At that size DPO will mostly memorise. Treat a before/after from
this notebook as evidence that the *pipeline* works end to end on real weights —
which is exactly what has never been shown — and not yet as evidence about how
much fine-tuning buys. Scaling the eval set is the next thing that changes a
conclusion.
